# Exercice : Vectorisation de la suite de Collatz (2025b)
![](https://deptinfo-ensip.univ-poitiers.fr/blank/TP_mini_vectorisation_collatz-sujet.ipynb:egarre06)

In [1]:
from IPython.display import HTML
HTML(url="https://ensip.gitlab.io/notebooks/data/nb2.css")

Vous vous souvenez de la conjecture de Collatz, Syracuse, $3n+1$,... ?
On prend un nombre de départ, par exemple : 5.
Puis s'il est pair, on le divise par 2. S'il est impair, on le multiplie par 3 et on ajoute 1.
Et on recommence.

La conjecture énonce qu'on finit toujours par tomber sur le nombre 1.
Le nombre d'étapes qu'il faut pour tomber sur 1 en partant de $n$ est la longueur du vol $n$.

La longueur du vol 1 est 0.
La longueur du vol 10 est 6 (10-> 5 -> 16 -> 8 -> 4 -> 2 -> 1)

Le programme suivant prend en paramètre un entier $n$, et renvoie la liste des longueurs des tous les vols de 1 à 10.

In [2]:
def longueurs_vols(n):
    def lg_vol(k):
        i = 0
        while k != 1:
            if k % 2 == 0:
                k = k // 2
            else:
                k = 3 * k + 1
            i = i + 1
        return i
    return [lg_vol(k) for k in range(1, n+1)]

In [3]:
longueurs_vols(10)

[0, 1, 7, 2, 5, 8, 16, 3, 19, 6]

Dans la suite de ce TP, en utilisant `numpy`, vous devrez réaliser une **version vectorisée** de cette fonction. Vous pourrez utiliser les masques, les opérateurs booléens sur les masques etc...

<div class="alert alert-danger">
Notez toutefois que ce calcul est juste prétexte à expérimenter sur la vectorisation. Dans le cas précis de la suite de Collatz, le gain observé (pour les grandes valeurs de n) n'est pas énorme car la plupart des suites se terminent assez rapidement.
</div>

L'idée est de mener «en parallèle» le calcul de la suite qui commence sur 1, de la suite qui commence sur 2, de la suite qui commence sur 3 etc... Ça donnerait ceci :

~~~
1  2  3  4  5  6 ...
1  1 10  2 16  3 ...
1  1  5  1  8 10 ...
1  1 16  1  4  5 ...
1  1  8  1  2 16 ...
1  1  4  1  1  8 ...
1  1  2  1  1  4 ...
1  1  1  1  1  2 ...
1  1  1  1  1  1 ...
~~~

La suite qui commence sur 3 est obtenue dans la 3e colonne.

On démarre donc avec le vecteur `1 2 3 4 5 6 ...`.
Puis on applique à ce vecteur (à chaque composante) la transformation : 

- si le coefficient vaut 1, on ne fait rien, 
- sinon,
  - s'il est pair, on le divise par 2, et 
  - sinon, on le multiplie par 3 et on ajoute 1.

C'est cette opération de transformation qui est vectorielle.

Écrivez la fonction qui prend une ligne de coefficients et lui applique la transformation précédente.

Pour diviser par 2 toutes les valeurs paires d'un vecteur `u`, on peut par exemple écrire : 

~~~
maskpair = (u % 2 == 0)
u[maskpair] = u[maskpair] // 2
~~~

La fonction `iteration` sera la fonction vectorisée. Elle ne contiendra pas de boucle (sinon, c'est que vous l'avez mal faite).

In [19]:
# Votre code ici
import numpy as np

def iteration(u):
    maskpair = (u%2==0)
    maskimpair = ((u%2==1) & (u!=1))
    u[maskpair]= u[maskpair]//2
    u[maskimpair]= u[maskimpair]*3+1
    return u

**Explication de la réponse ci-dessus :**
Cette fonction `iteration(u)` applique une étape de la suite de Collatz sur tout un tableau NumPy en parallèle. Elle utilise des 'masques booléens' (`maskpair` pour les pairs, `maskimpair` pour les impairs différents de 1) pour sélectionner et modifier simultanément les bons éléments du tableau sans utiliser de boucle `for` classique en Python, ce qui est beaucoup plus rapide (vectorisation).

In [22]:
# Testez votre fonction 
iteration(np.array([1, 2, 3, 4, 5, 6]))

array([ 1,  1, 10,  2, 16,  3])

La fonction doit satisfaire les tests suivants :

In [23]:
assert iteration(np.array([4])) == np.array([2])
assert iteration(np.array([3])) == np.array([10])
assert iteration(np.array([1])) == np.array([1])
assert np.all(iteration(np.array([1, 2, 3, 4, 5, 6])) == np.array([1, 1, 10, 2, 16, 3]))
assert np.all(iteration(np.array([1, 1, 10, 2, 16, 3])) == np.array([1, 1, 5, 1, 8, 10]))

À présent, écrivez une procédure qui prend en paramètre un entier `n`, construit
le vecteur `[1, 2, 3, ..., n]`, et calcule les itérations (avec la fonction précédente)
jusquà ce que toutes les valeurs du vecteur soient égales à 1.
Cette fonction contiendra une boucle, et à chaque tour, vous afficherez le vecteur calculé.

L'exécution de `vols(6)` devra produire quelquechose comme ceci :

~~~
[ 1  1 10  2 16  3]
[ 1  1  5  1  8 10]
[ 1  1 16  1  4  5]
[ 1  1  8  1  2 16]
[1 1 4 1 1 8]
[1 1 2 1 1 4]
[1 1 1 1 1 2]
[1 1 1 1 1 1]
~~~

<div class="alert alert-danger">
    Attention lorsque vous testerez ! L'exécution peut potentiellement bloquer.
Si le calcul ne se termine pas, pensez à appuyer sur Stop rapidement
dans la barre d'icones : ■
</div>

In [58]:
# Votre code ici
def vols(n):
    ite = np.arange(1,n+1) 
    while np.any(ite!=1) == True:
        ite = iteration(ite)
        print(ite)
    


**Explication de la réponse ci-dessus :**
Cette réponse partielle sert à itérer et afficher les étapes de la suite de Collatz de façon vectorisée. Elle crée un tableau `ite` allant de 1 à n, et utilise une boucle `while` qui continue tant qu'au moins un élément n'a pas atteint 1 (grâce à `np.any(ite!=1)`). Elle applique la fonction `iteration` en boucle.

In [59]:
# Testez votre fonction
# Attention... cette fonction peut potentiellement bloquer. 
# Si le calcul ne se termine pas, pensez à appuyer sur Stop rapidement
# dans la barre d'icones : ■

vols(6)

[ 1  1 10  2 16  3]
[ 1  1  5  1  8 10]
[ 1  1 16  1  4  5]
[ 1  1  8  1  2 16]
[1 1 4 1 1 8]
[1 1 2 1 1 4]
[1 1 1 1 1 2]
[1 1 1 1 1 1]


Nous devons finalement calculer la longueur des vols. Pour faire ça, nous allons initialiser un vecteur à 0, contenant 
autant de valeurs que notre vecteur qui sert au calcul de la suite.

~~~
calcul         compteur
[1 2 3 4 5 6]  [0 0 0 0 0 0]
~~~

Puis au début de chaque itération, nous augmentons de 1 chaque case du compteur qui correspond à une case non égale à 1 dans le vecteur de calcul. Puis nous modifions le vecteur de calcul comme précédemment. Ça donne ceci :

~~~
     calcul              compteur
                      [0 0 0 0 0 0]
[1  2  3  4  5  6]    
                      [0 1 1 1 1 1]
[1  1 10  2  16  3]    
                      [0 1 2 2 2 2]
[1  1  5  1  8  10]    
                      [0 1 3 2 3 3]
[1  1 16  1  4   5]    
                      [0 1 4 2 4 4]
...


~~~

À la fin, lorsque `calcul` ne contiendra plus que des `1`, `compteur` contiendra la longueur de chaque vol.

Réécrivez la procédure `vol` (ce sera maintenant une fonction) pour qu'elle calcule aussi ce compteur et le renvoie à la fin.



In [61]:
# Votre code ici
def vols(n):
    calcul = np.arange(1,n+1) 
    compteur = np.zeros(n,dtype=int)
    while np.any(calcul!=1) == True:
        compteur[(calcul!=1)] = compteur[(calcul!=1)] + 1
        calcul = iteration(calcul)
        

    return compteur

    

**Explication de la réponse ci-dessus :**
Cette fonction `vols(n)` améliorée calcule les longueurs de vols pour tous les nombres de 1 à n en même temps. Elle initialise un tableau de zéros `compteur`. Tant qu'il reste des éléments différents de 1, elle incrémente le compteur pour ces éléments spécifiques et passe à l'itération suivante via `iteration()`. Elle retourne finalement le tableau des longueurs.

In [62]:
# Testez votre fonction pour n = 10, vous devez obtenir la même chose qu'avec le code donné en début de TP
# [0,  1,  7,  2,  5,  8, 16,  3, 19,  6]
vols(10)

array([ 0,  1,  7,  2,  5,  8, 16,  3, 19,  6])